In [1]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from matplotlib import cm
import pandas as pd
import plotly.graph_objects as go
from itertools import product

In [2]:
z = np.array([[1, 2], [3, 4], [5, 6], [7, 8]])
means = np.array([[-3, 0], [3, 0], [2, -4]])
z[np.newaxis, 1] - means

array([[6, 4],
       [0, 4],
       [1, 8]])

In [3]:
np.array(
    [np.linalg.norm(z[np.newaxis, i] - means, axis=1) ** 2 for i in range(z.shape[0])]
)

array([[ 20.,   8.,  37.],
       [ 52.,  16.,  65.],
       [100.,  40., 109.],
       [164.,  80., 169.]])

In [4]:
z[np.newaxis, 1].shape

(1, 2)

In [5]:
means.shape

(3, 2)

In [6]:
def compute_class_probs(z, means, kappa, M, beta=0.5):
    K = means.shape[0]
    distance = np.array(
        [
            np.linalg.norm(z[np.newaxis, i] - means, axis=1) ** 2
            for i in range(z.shape[0])
        ]
    )
    p_z_c = ((1 - beta) / K) * (kappa / (2 * np.pi)) * np.exp(-kappa * distance)
    p_z_oog = np.array([beta / (M**2)])
    p_z_oog = np.broadcast_to(p_z_oog, (p_z_c.shape[0], 1))
    p_z_c = np.concatenate([p_z_c, p_z_oog], axis=1)
    p_z = np.sum(p_z_c, axis=1)
    p_c_z = p_z_c / p_z[:, np.newaxis]
    return p_c_z

In [41]:
# Параметры
means = np.array([[-3, 0], [3, 0], [-1.8, 1.8]])
false_accept_point = [-2.428, -1.054]
class_labels = np.arange(means.shape[0])
size = 300
value_range = 6
kappa = 1
x_shift = 0
y_shift = 1

# Сетка
x = np.linspace(-value_range, value_range, size)
y = np.linspace(-value_range + y_shift, value_range - y_shift, size)
grid_x, grid_y = np.meshgrid(x, y, indexing="ij")
grid_product = np.array(list(product(x, y)))

# Вычисление вероятностей и неопределенности
p_c_z = compute_class_probs(grid_product, means, kappa, value_range * 2)
uncertainty = -np.sum(
    p_c_z * np.log(p_c_z + 1e-15), axis=1
)  # Добавлен эпсилон против log(0)
z = uncertainty.reshape((size, size))

# Создание фигуры
fig = go.Figure()

# Контурный график неопределенности
fig.add_trace(
    go.Contour(
        x=x,
        y=y,
        z=z.T,
        colorscale="Blues",  # аналог cm.Blues
        contours=dict(
            showlabels=False,
            coloring="fill",
            start=z.min(),  # начальное значение
            end=z.max(),  # конечное значение
            size=(z.max() - z.min()) / 20,
        ),
        line=dict(width=0),
        showscale=False,
        # hoverinfo='skip',
    )
)

# copute decision boundary

eps = 0  # 1e-2
boundary = np.max(p_c_z[:, :-1], axis=1) - p_c_z[:, -1]  # < eps
boundary = boundary.reshape((size, size))

fig.add_trace(
    go.Contour(
        x=x,
        y=y,
        z=boundary.T,
        contours=dict(
            coloring="lines",  # только линии, без заливки
            start=eps,
            end=eps,
            size=eps,  # один уровень
        ),
        line=dict(width=3, color="white"),
        showscale=False,
        showlegend=True,
        hoverinfo="skip",
        # name=f'Uncertainty = {eps}'
    )
)

# fig.add_trace(go.Contour(
#     x=x,
#     y=y,
#     z=boundary.T,
#     contours=dict(
#         type='constraint',
#         operation='<',        # Показываем область, где z < threshold
#         value=eps
#     ),
#     line=dict(width=1, color='white'),  # стильно выделить
#     showlegend=True,
#     name=f'Uncertainty < {eps}',
#     hoverinfo='skip',
#     showscale=False,
# ))


# Точки центров классов
fig.add_trace(
    go.Scatter(
        x=means[:, 0],
        y=means[:, 1],
        mode="markers",
        marker=dict(size=12, color=class_labels, line=dict(width=1, color="black")),
        name="Class Centers",
    )
)

# Дополнительная точка (0,0)
fig.add_trace(
    go.Scatter(
        x=[false_accept_point[0]],
        y=[false_accept_point[1]],
        mode="markers",
        marker=dict(size=10, color="red", symbol="star"),
        name="False Accept Point",
    )
)

# Настройки вида
fig.update_layout(
    title=None,
    xaxis_title=None,
    yaxis_title=None,
    xaxis=dict(scaleanchor="y", scaleratio=1, visible=False),
    yaxis=dict(visible=False),
    showlegend=False,  # Убираем легенду, как в оригинале
    width=600,
    height=600,
    margin=dict(l=20, r=20, t=20, b=20),
    plot_bgcolor="white",
    paper_bgcolor="white",
    hovermode="closest",
)

# Убираем рамку (spines)
fig.update_xaxes(showline=False, zeroline=False)
fig.update_yaxes(showline=False, zeroline=False)

# Сохранение как PNG через kaleido (нужно установить: pip install kaleido)
# fig.show()
# fig.write_image("test.png", format="png", width=600, height=600, scale=5)  # dpi ~300 при scale=5

# При необходимости: сохранение в PDF
# fig.write_image("test.pdf", format="pdf", width=600, height=600)

# Отображение (в Jupyter или интерактивной среде)
fig  # .show()

In [8]:
import kaleido

print(kaleido.version)

AttributeError: module 'kaleido' has no attribute 'version'

In [ ]:
np.sum(p_c_z, axis=1).max()

1.0000000000000002